# Query Public Data with OData

Reads live data from [opteryx.app](https://opteryx.app), the hosted service, through its [OData v4](https://www.odata.org/) feed, using nothing more than `requests` and pandas. It works through the public **National Vulnerability Database (NVD)** and **CISA Known Exploited Vulnerabilities (KEV)** datasets: discovering what's there, filtering and sorting server-side, paging through a large result, and aggregating with `$apply`.

This notebook accompanies [Querying via OData](https://docs.opteryx.app/docs/guides/querying-via-odata) on docs.opteryx.app.

**No account needed.** The `public.security` and `public.geopolitics` collections allow anonymous reads over OData, so every cell runs without a token. The last section shows how to add one for datasets that aren't public.

## 1. Install

Colab already has both packages; this line is for running the notebook elsewhere.

In [ ]:
%pip install -q requests pandas matplotlib

## 2. A small helper

Every request is a `GET` against `https://odata.opteryx.app/api/v4/`. The helper below takes a path and the query options as keyword arguments, adds the `$` to each option name, and prints the URL it requested so you can see exactly what was sent.

A failed read is never a `200` with an empty result: the service returns a non-2xx status with a JSON body shaped `{"error": {"code": ..., "message": ...}}`. The helper raises that message as an exception.

In [ ]:
from urllib.parse import unquote, urljoin

import pandas as pd
import requests

HOST = "https://odata.opteryx.app"
BASE = HOST + "/api/v4/"

session = requests.Session()


class ODataError(Exception):
    pass


def fetch(path, **options):
    """GET a path (or a full URL) under the service root; `top=5` is sent as `$top=5`."""
    params = {f"${name}": str(value).lower() if isinstance(value, bool) else value for name, value in options.items()}
    response = session.get(urljoin(BASE, path), params=params, timeout=60)
    print("GET", unquote(response.url).replace("+", " "))
    if not response.ok:
        raise ODataError(f"{response.status_code}: {response.json()['error']['message']}")
    return response


def odata(path, **options):
    return fetch(path, **options).json()

## 3. Discover datasets

The service document at the root lists every EntitySet your credentials can read. With no token, that's the public datasets. Each entry's `url` is the path to query, shaped `{workspace}/{collection}/{dataset}`, and `source` says whether it's a table, a view, or something else.

In [ ]:
service = odata("")
datasets = pd.DataFrame(service["value"])
datasets[["name", "url", "source", "role"]]

## 4. Read a dataset's schema from `$metadata`

`$metadata` is the CSDL document that Power BI and Excel read to learn column names and types. Request it for a single dataset by adding `/$metadata` to its path. It's XML, so parse it with the standard library.

In [ ]:
import xml.etree.ElementTree as ET

EDM = "{http://docs.oasis-open.org/odata/ns/edm}"

metadata = ET.fromstring(fetch("public/security/nvd_vulnerabilities/$metadata").content)

columns = pd.DataFrame(
    [
        {"column": prop.get("Name"), "type": prop.get("Type"), "nullable": prop.get("Nullable")}
        for prop in metadata.iter(f"{EDM}Property")
    ]
)
columns

Alongside the columns, the metadata carries annotations about the dataset itself, such as its description, the snapshot you're reading and how many rows it holds:

In [ ]:
for annotation in metadata.iter(f"{EDM}Annotation"):
    term = annotation.get("Term")
    if term in ("Custom.Description", "Custom.Snapshot.Id", "Custom.Snapshot.TotalRecords", "Custom.LastModified", "Custom.Tags.Count"):
        print(f"{term:<30} {annotation.get('String') or annotation.get('Int')}")

## 5. Query a dataset

### `$top`

Limit the number of rows returned. Without `$top` you get the default page of 100 rows.

In [ ]:
first = odata("public/security/nvd_vulnerabilities", top=5)
pd.DataFrame(first["value"])

### `$select`

Return only the columns you need. The descriptions are long, so leaving them out keeps responses small.

In [ ]:
result = odata("public/security/nvd_vulnerabilities", select="cve_id,published_at,severity,cvss_score,vendor,product", top=5)
pd.DataFrame(result["value"])

### `$filter`

Restrict rows with an OData filter expression. Strings are in single quotes. This is the guide's example on `public.geopolitics.countries`:

In [ ]:
europe = odata("public/geopolitics/countries", filter="region eq 'Europe'", orderby="country_name_common", select="country_name_common,capital,subregion", top=5)
pd.DataFrame(europe["value"])

Dates and timestamps are written **unquoted**, in ISO 8601. Entries CISA has added to its known-exploited catalog since the start of 2025:

In [ ]:
kev_recent = odata(
    "public/security/cisa_kev",
    filter="date_added ge 2025-01-01",
    select="cve_id,vendor_project,product,vulnerability_name,date_added",
    top=5,
)
pd.DataFrame(kev_recent["value"])

Quoting the date makes it a string, and comparing a string to a `DATE` column is rejected with a `400` that says what to write instead:

In [ ]:
try:
    odata("public/security/cisa_kev", filter="date_added ge '2025-01-01'", top=5)
except ODataError as error:
    print(error)

`now()` and an ISO 8601 duration express a rolling window, evaluated on the server, so the query stays correct when you re-run it tomorrow. Critical CVEs published in the last 30 days:

In [ ]:
last_30_days = odata(
    "public/security/nvd_vulnerabilities",
    filter="severity eq 'CRITICAL' and published_at ge now() sub duration'P30D'",
    select="cve_id,published_at,cvss_score,vendor,product",
    orderby="published_at desc",
    top=10,
)
pd.DataFrame(last_30_days["value"])

### `$orderby`

Sort by one or more columns, each `asc` (the default) or `desc`. The highest-scoring CVEs, newest first among ties:

In [ ]:
top_scores = odata(
    "public/security/nvd_vulnerabilities",
    select="cve_id,published_at,cvss_score,vendor,product",
    orderby="cvss_score desc,published_at desc",
    top=10,
)
pd.DataFrame(top_scores["value"])

### `$count`

`$count=true` adds `@odata.count`, the number of rows the filter matched before `$top` was applied. With `$top=0` you get just the count and no rows:

In [ ]:
critical = odata("public/security/nvd_vulnerabilities", filter="severity eq 'CRITICAL'", count=True, top=0)
print(f"{critical['@odata.count']:,} critical CVEs")

## 6. Page through a result with `@odata.nextLink`

When a query matches more rows than `$top`, the response has exactly `$top` rows plus an `@odata.nextLink` pointing at the next page. Keep following it until a response comes back without one.

Two details from the guide matter here:

- The link is **relative**, so resolve it against `https://odata.opteryx.app`.
- The `$` in its query string arrives percent-encoded as `%24`. That's equivalent; pass the link through unchanged rather than rewriting it.

There are more than 30,000 critical CVEs, so this loop stops after `MAX_PAGES` pages. Remove the limit to read the whole result.

In [ ]:
MAX_PAGES = 3

page = odata(
    "public/security/nvd_vulnerabilities",
    filter="severity eq 'CRITICAL'",
    select="cve_id,published_at,cvss_score",
    orderby="published_at desc",
    top=500,
)
pages = [pd.DataFrame(page["value"])]

while "@odata.nextLink" in page and len(pages) < MAX_PAGES:
    print("  nextLink:", page["@odata.nextLink"])
    page = odata(urljoin(HOST, page["@odata.nextLink"]))
    pages.append(pd.DataFrame(page["value"]))

critical_pages = pd.concat(pages, ignore_index=True)
more = "more pages remain" if "@odata.nextLink" in page else "that was the last page"
print(f"Read {len(critical_pages):,} rows in {len(pages)} pages; {more}.")
critical_pages.head()

## 7. Aggregate server-side with `$apply`

`$apply` groups and aggregates on the server, so only the summary comes back. `$count`, `sum`, `average`, `min` and `max` are available. The guide's example counts countries per region:

In [ ]:
regions = odata("public/geopolitics/countries", apply="groupby((region),aggregate($count as country_count))")
pd.DataFrame(regions["value"])

The same pattern over all ~400,000 NVD records, counting CVEs and averaging their CVSS score per severity band. `$orderby` can sort by an aggregate alias on the server, but here the summary is sorted in pandas:

In [ ]:
by_severity = odata(
    "public/security/nvd_vulnerabilities",
    apply="groupby((severity),aggregate($count as cves,cvss_score with average as avg_cvss))",
)
pd.DataFrame(by_severity["value"]).sort_values("cves", ascending=False)

`filter()` inside `$apply` narrows the rows before grouping. The vendors with the most known-exploited vulnerabilities used in ransomware campaigns. One row comes back per vendor, so `$top` is raised above the default 100 to get every group in one page:

In [ ]:
ransomware = odata(
    "public/security/cisa_kev",
    apply="filter(known_ransomware_campaign_use eq 'Known')/groupby((vendor_project),aggregate($count as kevs))",
    top=1000,
)
by_vendor = pd.DataFrame(ransomware["value"]).sort_values("kevs", ascending=False).head(10)
by_vendor.plot.barh(x="vendor_project", y="kevs", legend=False, figsize=(8, 4), title="Known ransomware-linked KEVs by vendor").invert_yaxis()

## 8. Datasets that aren't public

Your own datasets need a bearer token on every request:

```
Authorization: Bearer <token>
```

It must be a JWT access token. A personal access token from Opteryx Studio (**Settings → API Tokens**) gives you a client ID and secret; exchange that pair for an access token at the [Authentication API](https://docs.opteryx.app/docs/reference/api/authentication-api), then send the access token. The raw client ID and secret aren't accepted by the OData service.

Keep the secret out of the notebook. In Colab, add `OPTERYX_CLIENT_SECRET` under the key icon in the left sidebar (Colab Secrets) and turn on notebook access; elsewhere, the cell asks for it in a hidden prompt. Set `CLIENT_ID` and `MY_DATASET` to run it; while `CLIENT_ID` is empty, the cell does nothing.

In [ ]:
from getpass import getpass

CLIENT_ID = ""    # from Settings → API Tokens
MY_DATASET = ""   # path from the service document, e.g. "acme/sales/orders"

if CLIENT_ID:
    try:
        from google.colab import userdata

        CLIENT_SECRET = userdata.get("OPTERYX_CLIENT_SECRET")
    except Exception:
        CLIENT_SECRET = getpass("Client secret: ")

    token = requests.post(
        "https://authenticate.opteryx.app/token",
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
        timeout=30,
    )
    token.raise_for_status()
    # Every later request from this session carries the token. It's short-lived: re-run this cell when it expires.
    session.headers["Authorization"] = f"Bearer {token.json()['access_token']}"

    mine = pd.DataFrame(odata("")["value"])
    display(mine[["name", "url", "source", "role"]])
    if MY_DATASET:
        display(pd.DataFrame(odata(MY_DATASET, top=10)["value"]))
else:
    print("Set CLIENT_ID (and optionally MY_DATASET), then run this cell again.")

## Things to know

- **Read-only.** OData here has no `INSERT`, `UPDATE` or `DELETE`.
- **Empty means empty.** A `200` with an empty `value` means the query matched no rows; any failure is a non-2xx status with an error body.
- **Not implemented.** `$search` and `$expand` return `501`.
- **From a shell.** Wrap the URL in single quotes so the shell doesn't expand `$top`, and encode spaces as `%20` (or use `curl -G --data-urlencode '$filter=...'`).
- **Power BI and Excel.** Point either at `https://odata.opteryx.app/api/v4/` or at a dataset URL; both follow `@odata.nextLink` for you.